In [ ]:
pip install requests pymupdf pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.0/20.0 MB 13.4 MB/s eta 0:00:00


In [ ]:
import pandas as pd
data = pd.read_excel('column.xlsx')
print(data.head())

   Id     Paper_id  Rate0  Rate1  Rate2  Avg_rate  Decision  \
0   1  DigrnXQNMTe      1      2      3  2.000000         0   
1   2  cL4wkyoxyDJ      1      2      2  1.666667         0   
2   3   fcZIsyzF6g      1      3      2  2.000000         0   
3   4  Iuq6u10sCdl      2      3      2  2.333333         0   
4   5  xPw-dr5t1RH      2      2      2  2.000000         0   

                                           Reviewer0  \
0  The works proposes a generalization of MMD-squ...   
1  The paper proposes a defense against adversari...   
2  The paper proposed an algorithm for decentrali...   
3  In this paper the authors address the mathemat...   
4  summary: This paper proposes a framework, KETG...   

                                           Reviewer1  \
0  Summary. The authors describe a family of kern...   
1  #################################### Summary #...   
2  The paper proposed a decentralized training al...   
3  The paper claims to provide theoretical guaran...   
4  T

In [ ]:
import fitz  # PyMuPDF
import requests
import pandas as pd
from io import BytesIO
import os
from multiprocessing import Pool, cpu_count

# Function to download PDF
def download_pdf(url):
    response = requests.get(url)
    if response.status_code == 200:
        return BytesIO(response.content)  # Return PDF content as a BytesIO object
    else:
        print(f"Failed to download PDF from {url}")
        return None

# Function to extract text and images from PDF
def extract_text_and_images_from_pdf(pdf_stream, output_image_dir="extracted_images"):
    try:
        doc = fitz.open(stream=pdf_stream, filetype="pdf")
        text = ""
        images = []

        # Create directory to save images
        os.makedirs(output_image_dir, exist_ok=True)

        for page_num in range(len(doc)):
            page = doc.load_page(page_num)
            # Extract text
            text += page.get_text("text")

            # Extract images
            image_list = page.get_images(full=True)
            for img_index, img in enumerate(image_list):
                xref = img[0]  # Image reference number
                base_image = doc.extract_image(xref)
                image_bytes = base_image["image"]  # Raw image data
                image_ext = base_image["ext"]  # Image extension (e.g., "png", "jpeg")
                image_path = os.path.join(output_image_dir, f"page_{page_num + 1}_img_{img_index + 1}.{image_ext}")
                with open(image_path, "wb") as image_file:
                    image_file.write(image_bytes)
                images.append(image_path)

        return text, images
    except Exception as e:
        print(f"Error extracting content from PDF: {e}")
        return None, None

# Worker function to process a single PDF link
def process_pdf_link(row):
    pdf_link = row['Link']
    print(f"Processing: {pdf_link}")
    pdf_stream = download_pdf(pdf_link)
    if pdf_stream:
        text, images = extract_text_and_images_from_pdf(pdf_stream)
        return {
            'Link': pdf_link,
            'Text': text,
            'Images': images
        }
    else:
        return {
            'Link': pdf_link,
            'Text': None,
            'Images': None
        }

# Main function to process all PDF links using multiprocessing
def process_pdf_links(df, num_processes=None):
    if num_processes is None:
        num_processes = cpu_count()  # Use all available CPU cores

    # Split the DataFrame into chunks for parallel processing
    with Pool(processes=num_processes) as pool:
        results = pool.map(process_pdf_link, df.to_dict('records'))

    # Convert results back to a DataFrame
    results_df = pd.DataFrame(results)
    return results_df

# Example DataFrame with PDF links

df = pd.DataFrame(data)

# Process all PDF links using multiprocessing
results_df = process_pdf_links(df)

# Display the updated DataFrame
print("\nUpdated DataFrame:")
print(results_df)

# Save the DataFrame to a CSV file
results_df.to_csv("pdf_text_and_images_extracted.csv", index=False)

Streaming output truncated to the last 5000 lines.
Processing: https://openreview.net/pdf?id=C1VUD8RZ5wq
Processing: https://openreview.net/pdf?id=W0KJGRBH60o
Processing: https://openreview.net/pdf?id=F8xpAPm_ZKS
Processing: https://openreview.net/pdf?id=ZB8vwY8cg6Y
Processing: https://openreview.net/pdf?id=8KhxoxKP3iL
Processing: https://openreview.net/pdf?id=oe8U8WETg4t
Processing: https://openreview.net/pdf?id=8TnLOVrNRNp
Processing: https://openreview.net/pdf?id=JRJTVcG0f-N
Processing: https://openreview.net/pdf?id=7Yhok3vJpU
Processing: https://openreview.net/pdf?id=D7hX1d3ov2c
Processing: https://openreview.net/pdf?id=DC1Im3MkGG
Processing: https://openreview.net/pdf?id=Uozyxz3eKY
Processing: https://openreview.net/pdf?id=drRnrGMZ3ze
Processing: https://openreview.net/pdf?id=qOCdZn3lQIJ
Processing: https://openreview.net/pdf?id=YYULSFvKru9
Processing: https://openreview.net/pdf?id=YhhEarKSli9
Processing: https://openreview.net/pdf?id=BvrKnFq_454
Processing: https://openreview.net

In [ ]:
print(results_df['Text'][0])

Under review as a conference paper at ICLR 2021
A
GENERALIZED
PROBABILITY
KERNEL
ON
DIS-
CRETE DISTRIBUTIONS AND ITS APPLICATION IN TWO-
SAMPLE TEST
Anonymous authors
Paper under double-blind review
ABSTRACT
We propose a generalized probability kernel(GPK) on discrete distributions with
ﬁnite support. This probability kernel, deﬁned as kernel between distributions in-
stead of samples, generalizes the existing discrepancy statistics such as maximum
mean discrepancy(MMD) as well as probability product kernels, and extends to
more general cases. For both existing and newly proposed statistics, we estimate
them through empirical frequency and illustrate the strategy to analyze the re-
sulting bias and convergence bounds. We further propose power-MMD, a natural
extension of MMD in the framework of GPK, illustrating its usage for the task
of two-sample test. Our work connects the ﬁelds of discrete distribution-property
estimation and kernel-based hypothesis test, which might shed light on m

In [ ]:
import pandas as pd

# Load the DataFrames
text_data = pd.read_csv('pdf_text_and_images_extracted.csv')
everything_else = pd.read_excel('column.xlsx')

# Join the DataFrames on the 'Link' column
merged_df = pd.merge(text_data, everything_else, on='Link', how='inner')  # Use 'inner' to keep only matching rows

# Display the merged DataFrame
print(merged_df.head())

# Save the merged DataFrame to a new file (optional)
merged_df.to_csv('merged_data.csv', index=False)

                                        Link  \
0  https://openreview.net/pdf?id=DigrnXQNMTe   
1  https://openreview.net/pdf?id=cL4wkyoxyDJ   
2   https://openreview.net/pdf?id=fcZIsyzF6g   
3  https://openreview.net/pdf?id=Iuq6u10sCdl   
4  https://openreview.net/pdf?id=xPw-dr5t1RH   

                                                Text  \
0  Under review as a conference paper at ICLR 202...   
1  Under review as a conference paper at ICLR 202...   
2  Under review as a conference paper at ICLR 202...   
3  Under review as a conference paper at ICLR 202...   
4  Under review as a conference paper at ICLR 202...   

                                              Images  Id     Paper_id  Rate0  \
0             ['extracted_images/page_14_img_1.png']   1  DigrnXQNMTe      1   
1                                                 []   2  cL4wkyoxyDJ      1   
2  ['extracted_images/page_4_img_1.png', 'extract...   3   fcZIsyzF6g      1   
3                                                 []  

KeyboardInterrupt: 

In [ ]:
print(len(merged_df))
print(len(results_df))

6283
6283


In [ ]:
# Load model directly
from transformers import AutoModel
model = AutoModel.from_pretrained("allenai/scibert_scivocab_uncased")